<a href="https://colab.research.google.com/github/ShahendaWael/DEPI_Project/blob/main/Session1_AgenticAI_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Session 1: Agentic AI Concepts & Architecture — Hands-On Lab
### Generative & Agentic AI Course

---

##  What You Will Build Today

In this lab, you will go from **understanding** Agentic AI to **building** one:

| Phase | What You'll Do | Time |
|-------|---------------|------|
| Phase 1 | Explore LLM vs Agent — see the difference live | 15 min |
| Phase 2 | Build the Agent Loop (Perceive → Reason → Act → Learn) | 25 min |
| Phase 3 | Add Tools — make your agent actually DO things | 20 min |
| Phase 4 | Multi-Agent System — two agents collaborating | 15 min |
| Phase 5 | Mini Challenge — build your own agent! | 15 min |

---

>  **Key Idea from the lecture:** Agentic AI = LLMs with the ability to act autonomously.
> An agent doesn't just *generate* text — it *understands*, *plans*, *acts*, and *learns*.

---

## 🔧 Phase 1: Setup & LLM vs Agent Comparison

### 1.1 Install & Import

We'll use the **Gemini API** (Google's LLM) as the brain of our agent.

In [ ]:
# Install the Google Generative AI library
!pip install -q google-generativeai

import google.generativeai as genai
import json
import time
import random
from datetime import datetime

print("✅ All libraries installed and imported!")

### 1.2 Configure Your API Key

Get your free Gemini API key from: https://aistudio.google.com/apikey

> ⚠️ **Never share your API key publicly!**

In [ ]:
from google.colab import userdata

# Option 1: Use Colab Secrets (recommended)
# Go to 🔑 icon in left sidebar → Add secret named 'GEMINI_API_KEY'
try:
    API_KEY = userdata.get('GEMINI_API_KEY')
except:
    # Option 2: Paste directly (for testing only)
    API_KEY = "YOUR_API_KEY_HERE"  # ← Replace this

genai.configure(api_key=API_KEY)
model = genai.GenerativeModel('gemini-1.5-flash')

# Quick test
response = model.generate_content("Say 'Hello Agent World!' in one line")
print(response.text)
print("\n✅ Gemini is working!")

### 1.3 🧪 Experiment: LLM vs Agent — What's the Difference?

Let's see why a plain LLM is **not** an agent.

**Scenario:** A user asks: *"What's the weather in Cairo and should I take an umbrella?"*

#### Step A: Ask the raw LLM (no tools, no planning)

In [ ]:
# ============================================
# RAW LLM — Just generates text, no real action
# ============================================

user_question = "What's the weather in Cairo right now and should I take an umbrella?"

response = model.generate_content(user_question)
print("🗣️ Raw LLM Response:")
print(response.text)
print()
print(" Notice: The LLM CANNOT check real weather!")
print("   It can only generate text based on its training data.")
print("   It might give outdated or hallucinated weather info.")

#### Step B: Now see what an AGENT would do

An agent would:
1. **Perceive** the request → understand user wants weather info
2. **Reason** → decide it needs a weather tool
3. **Act** → call the weather API
4. **Learn** → combine the real data with reasoning to answer

We'll build this step by step! But first, let's simulate it:

In [ ]:
# ============================================
# SIMULATED AGENT — Has tools and can plan
# ============================================

# Simulated weather tool (in real life, this calls an API)
def get_weather(city):
    """Simulates a weather API call"""
    weather_db = {
        "cairo": {"temp": 35, "condition": "Sunny", "humidity": 20, "rain_chance": 5},
        "london": {"temp": 12, "condition": "Rainy", "humidity": 85, "rain_chance": 80},
        "tokyo": {"temp": 22, "condition": "Cloudy", "humidity": 60, "rain_chance": 30},
    }
    return weather_db.get(city.lower(), {"error": f"No data for {city}"})

# Agent workflow
print("🤖 AGENT WORKFLOW:")
print("=" * 50)

# Step 1: Perceive
print("\n👁️ Step 1 — PERCEPTION:")
print(f"   User asked: '{user_question}'")
print("   → Detected intent: weather query")
print("   → Detected city: Cairo")

# Step 2: Reason
print("\n🧠 Step 2 — REASONING:")
print("   → I need real weather data, not my training knowledge")
print("   → Available tool: get_weather()")
print("   → Decision: Call get_weather('cairo')")

# Step 3: Act
print("\n⚡ Step 3 — ACTION:")
weather_data = get_weather("cairo")
print(f"   → Called get_weather('cairo')")
print(f"   → Result: {json.dumps(weather_data, indent=2)}")

# Step 4: Respond with real data
print("\n💬 Step 4 — RESPONSE:")
agent_prompt = f"""Based on this REAL weather data: {json.dumps(weather_data)}
Answer the user's question: {user_question}
Be helpful and give a clear recommendation."""

response = model.generate_content(agent_prompt)
print(f"   {response.text}")

print("\n" + "=" * 50)
print("✅ See the difference? The agent used REAL data, not guesses!")

### 📝 Reflection Question

**TODO:** In the cell below, write your answer:

*What are the 3 main differences you noticed between the raw LLM response and the agent response?*

In [ ]:
# TODO: Write your answer here as comments
# Difference 1:
# Difference 2:
# Difference 3:


---

## 🔄 Phase 2: Build the Agent Loop

From the lecture, we learned agents follow this cycle:

**Perceive → Reason → Act → Learn**

Let's build each component!

### 2.1 Define Our Tools

An agent needs tools to interact with the world. Let's create a toolkit:

In [ ]:
# ============================================
# TOOL DEFINITIONS — The agent's capabilities
# ============================================

def calculator(expression):
    """Evaluates a math expression safely"""
    try:
        # Only allow safe math operations
        allowed = set('0123456789+-*/.() ')
        if all(c in allowed for c in expression):
            result = eval(expression)
            return {"result": result, "status": "success"}
        else:
            return {"error": "Invalid expression", "status": "failed"}
    except Exception as e:
        return {"error": str(e), "status": "failed"}

def search_knowledge(query):
    """Simulates a knowledge base search"""
    knowledge_base = {
        "python": "Python is a high-level programming language created by Guido van Rossum in 1991.",
        "machine learning": "ML is a subset of AI that enables systems to learn from data without being explicitly programmed.",
        "deep learning": "Deep learning uses neural networks with many layers to learn representations of data.",
        "transformer": "Transformers are neural network architecture using self-attention, introduced in 'Attention Is All You Need' (2017).",
        "langchain": "LangChain is an open-source framework for building LLM-powered applications.",
        "crewai": "CrewAI is a multi-agent framework where agents have roles, goals, and collaborate on tasks.",
    }
    query_lower = query.lower()
    results = []
    for key, value in knowledge_base.items():
        if key in query_lower or query_lower in key:
            results.append(value)
    if results:
        return {"results": results, "status": "success"}
    return {"results": ["No relevant information found."], "status": "no_match"}

def get_current_time():
    """Returns the current date and time"""
    now = datetime.now()
    return {"time": now.strftime("%Y-%m-%d %H:%M:%S"), "day": now.strftime("%A")}

def task_planner(goal):
    """Uses the LLM to break down a goal into steps"""
    prompt = f"""Break down this goal into 3-5 actionable steps.
Goal: {goal}
Return ONLY a JSON list of strings. Example: ["step 1", "step 2", "step 3"]
No extra text."""
    response = model.generate_content(prompt)
    try:
        steps = json.loads(response.text.strip().replace('```json', '').replace('```', ''))
        return {"steps": steps, "status": "success"}
    except:
        return {"steps": [response.text], "status": "raw_response"}

# Register all tools in a dictionary
TOOLS = {
    "calculator": {
        "function": calculator,
        "description": "Evaluates math expressions. Input: a math expression string."
    },
    "search_knowledge": {
        "function": search_knowledge,
        "description": "Searches a knowledge base for information. Input: a search query."
    },
    "get_current_time": {
        "function": get_current_time,
        "description": "Returns the current date and time. No input needed."
    },
    "task_planner": {
        "function": task_planner,
        "description": "Breaks down a complex goal into actionable steps. Input: a goal description."
    },
    "get_weather": {
        "function": get_weather,
        "description": "Gets weather data for a city. Input: city name."
    }
}

print("🧰 Agent Toolkit:")
for name, tool in TOOLS.items():
    print(f"   🔧 {name}: {tool['description']}")
print(f"\n✅ {len(TOOLS)} tools registered!")

### 2.2 Build the Perception Module

The agent first needs to **understand** what the user wants.

This is where the LLM shines — it interprets natural language and extracts:
- The **intent** (what the user wants to do)
- The **entities** (key information like city names, numbers)
- The **tool** needed

In [ ]:
# ============================================
# PERCEPTION — Understanding user intent
# ============================================

def perceive(user_input):
    """
    Uses the LLM to understand user intent and select the right tool.
    Returns a structured decision.
    """
    tool_descriptions = "\n".join(
        [f"- {name}: {info['description']}" for name, info in TOOLS.items()]
    )

    prompt = f"""You are an AI agent's perception module.
Analyze the user's request and decide which tool to use.

Available tools:
{tool_descriptions}
- none: If you can answer directly without any tool.

User request: "{user_input}"

Respond ONLY with this JSON (no extra text, no markdown):
{{
    "intent": "brief description of what user wants",
    "tool": "tool_name or none",
    "tool_input": "the input to pass to the tool, or empty string",
    "confidence": "high/medium/low"
}}"""

    response = model.generate_content(prompt)
    try:
        text = response.text.strip().replace('```json', '').replace('```', '').strip()
        return json.loads(text)
    except:
        return {
            "intent": "unclear",
            "tool": "none",
            "tool_input": "",
            "confidence": "low"
        }

# Test the perception module
test_queries = [
    "What is 245 * 18 + 32?",
    "Tell me about transformers in AI",
    "What time is it?",
    "What's the weather in London?",
    "Help me plan a machine learning project"
]

print("🧪 Testing Perception Module:")
print("=" * 60)
for query in test_queries:
    result = perceive(query)
    print(f"\n📝 Query: '{query}'")
    print(f"   Intent: {result.get('intent', 'N/A')}")
    print(f"   Tool:   {result.get('tool', 'N/A')}")
    print(f"   Input:  {result.get('tool_input', 'N/A')}")
    print(f"   Confidence: {result.get('confidence', 'N/A')}")
    time.sleep(1)  # Rate limit

print("\n✅ Perception module working!")

### 2.3 Build the Reasoning & Action Module

Now we connect perception → tool execution → response generation.

**TODO:** Complete the `execute_action` function below. Fill in the missing parts!

In [ ]:
# ============================================
# REASONING & ACTION — Execute the plan
# ============================================

def execute_action(perception_result):
    """
    Takes the perception output and executes the appropriate tool.
    Returns the tool result.
    """
    tool_name = perception_result.get("tool", "none")
    tool_input = perception_result.get("tool_input", "")

    print(f"\n⚡ ACTION: Executing tool '{tool_name}'")

    # TODO: Complete this function!
    # If tool_name is "none", return a dict with status "no_tool_needed"
    # If tool_name exists in TOOLS dict, call the function with tool_input
    # If tool_name is not found, return an error dict

    if tool_name == "none":
        # TODO: Return {"status": "no_tool_needed", "message": "..."}
        pass  # ← Replace this

    elif tool_name in TOOLS:
        # TODO: Get the function from TOOLS[tool_name]["function"]
        #       Call it with tool_input (or no args if tool is get_current_time)
        #       Return the result
        pass  # ← Replace this

    else:
        # TODO: Return {"error": f"Unknown tool: {tool_name}", "status": "failed"}
        pass  # ← Replace this


# Don't run this yet — complete the TODO first!

### 2.4 Build the Response Generator (Learning Step)

After executing a tool, the agent needs to **synthesize** the result into a helpful response.

This is also where the agent "learns" — it uses tool results to give grounded answers.

In [ ]:
# ============================================
# RESPONSE GENERATION — Synthesize final answer
# ============================================

def generate_response(user_input, perception, tool_result):
    """
    Combines the user question + tool result to generate a final response.
    """
    if tool_result.get("status") == "no_tool_needed":
        # Just answer directly
        response = model.generate_content(user_input)
        return response.text

    # Build a grounded prompt with real data
    prompt = f"""You are a helpful AI assistant. Answer the user's question using the data below.

User Question: {user_input}

Tool Used: {perception.get('tool', 'none')}
Tool Result: {json.dumps(tool_result, indent=2)}

Instructions:
- Use the tool result data to give an accurate answer
- Be concise and helpful
- If the tool returned an error, acknowledge it honestly
"""

    response = model.generate_content(prompt)
    return response.text

print("✅ Response generator defined!")

### 2.5 🏗️ Assemble the Complete Agent!

Now let's wire everything together into a single `Agent` class.

This is the **agentic loop** from the lecture:
1. Perceive → 2. Reason → 3. Act → 4. Respond

In [ ]:
# ============================================
# THE COMPLETE AGENT CLASS
# ============================================

class SimpleAgent:
    def __init__(self, name="Agent-1"):
        self.name = name
        self.memory = []  # Stores conversation history
        self.action_log = []  # Stores all actions taken

    def run(self, user_input):
        """Main agent loop: Perceive → Reason → Act → Respond"""
        print(f"\n{'='*60}")
        print(f"🤖 {self.name} Processing: '{user_input}'")
        print(f"{'='*60}")

        # Step 1: PERCEIVE
        print("\n👁️ STEP 1 — PERCEPTION")
        perception = perceive(user_input)
        print(f"   Intent: {perception.get('intent')}")
        print(f"   Tool selected: {perception.get('tool')}")
        print(f"   Confidence: {perception.get('confidence')}")

        # Step 2: REASON & ACT
        print("\n🧠 STEP 2 — REASONING & ACTION")
        tool_result = execute_action(perception)
        print(f"   Tool result: {json.dumps(tool_result, indent=2)[:200]}...")

        # Step 3: RESPOND
        print("\n💬 STEP 3 — GENERATING RESPONSE")
        final_response = generate_response(user_input, perception, tool_result)

        # Step 4: REMEMBER (Learning)
        self.memory.append({
            "query": user_input,
            "perception": perception,
            "tool_result": tool_result,
            "response": final_response[:200]
        })
        self.action_log.append({
            "time": datetime.now().strftime("%H:%M:%S"),
            "tool": perception.get('tool'),
            "status": tool_result.get('status', 'unknown')
        })

        print(f"\n📋 FINAL ANSWER:")
        print(f"{'─'*40}")
        print(final_response)
        print(f"{'─'*40}")
        print(f"📝 Memory size: {len(self.memory)} interactions stored")

        return final_response

    def show_memory(self):
        """Display the agent's memory"""
        print(f"\n🧠 {self.name}'s Memory ({len(self.memory)} entries):")
        for i, mem in enumerate(self.memory, 1):
            print(f"   {i}. Q: {mem['query'][:50]}... → Tool: {mem['perception'].get('tool')}")

    def show_action_log(self):
        """Display all actions taken"""
        print(f"\n📜 {self.name}'s Action Log:")
        for log in self.action_log:
            print(f"   [{log['time']}] Tool: {log['tool']} | Status: {log['status']}")

print("✅ Agent class defined! Ready to create an agent.")

### 🧪 Test Your Agent!

⚠️ **Make sure you completed the `execute_action` TODO above before running this!**

In [ ]:
# Create and test the agent
agent = SimpleAgent(name="Atlas")

# Test with different types of queries
test_queries = [
    "What is 1024 * 768?",
    "Tell me about machine learning",
    "What time is it right now?",
    "What's the weather in Tokyo?",
]

for query in test_queries:
    agent.run(query)
    time.sleep(2)  # Rate limit
    print("\n")

In [ ]:
# View the agent's memory and action log
agent.show_memory()
print()
agent.show_action_log()

---

## 🗺️ Phase 3: Multi-Step Planning Agent

From the lecture: *"An AI Agent breaks down tasks, selects the best tools, executes actions, reviews results, and adjusts its strategy."*

Let's build an agent that can handle **complex multi-step requests**!

**TODO:** Complete the `plan_and_execute` function below.

In [ ]:
# ============================================
# MULTI-STEP PLANNING AGENT
# ============================================

def plan_and_execute(complex_goal):
    """
    For complex queries, the agent:
    1. Breaks the goal into sub-tasks
    2. Executes each sub-task
    3. Synthesizes all results
    """
    print(f"\n{'='*60}")
    print(f"🎯 COMPLEX GOAL: {complex_goal}")
    print(f"{'='*60}")

    # Step 1: Break down into sub-tasks using the LLM
    print("\n📋 Step 1: Decomposing into sub-tasks...")
    planning_prompt = f"""Break this complex request into 2-4 simple sub-tasks.
Each sub-task should be something a single tool can handle.

Available tools: calculator, search_knowledge, get_current_time, get_weather, task_planner

Complex request: "{complex_goal}"

Return ONLY a JSON list of objects:
[
  {{"subtask": "description", "tool": "tool_name", "input": "tool input"}},
  ...
]
No extra text."""

    response = model.generate_content(planning_prompt)
    try:
        text = response.text.strip().replace('```json', '').replace('```', '').strip()
        subtasks = json.loads(text)
    except:
        print("   ⚠️ Planning failed, treating as single task")
        subtasks = [{"subtask": complex_goal, "tool": "none", "input": ""}]

    for i, task in enumerate(subtasks, 1):
        print(f"   Sub-task {i}: {task.get('subtask', 'N/A')} → Tool: {task.get('tool', 'N/A')}")

    # Step 2: Execute each sub-task
    print("\n⚡ Step 2: Executing sub-tasks...")
    results = []

    # TODO: Loop through subtasks and execute each one
    # For each subtask:
    #   - Get the tool name and input
    #   - If the tool exists in TOOLS, call it
    #   - Store the result in the 'results' list
    #   - Print the result for each sub-task

    for i, task in enumerate(subtasks, 1):
        tool_name = task.get("tool", "none")
        tool_input = task.get("input", "")

        # TODO: Complete this loop
        # Hint: check if tool_name is in TOOLS, then call TOOLS[tool_name]["function"]
        # Don't forget: get_current_time takes no arguments!
        pass  # ← Replace this

    # Step 3: Synthesize results
    print("\n🔗 Step 3: Synthesizing results...")
    synthesis_prompt = f"""Combine these sub-task results into a single helpful answer.

Original request: {complex_goal}

Sub-task results:
{json.dumps(results, indent=2)}

Give a clear, combined answer."""

    final = model.generate_content(synthesis_prompt)
    print(f"\n📋 FINAL COMBINED ANSWER:")
    print(f"{'─'*40}")
    print(final.text)
    return final.text

print("✅ Multi-step planner defined!")

In [ ]:
# Test the multi-step agent
plan_and_execute(
    "I want to know the weather in Cairo and London, then calculate the temperature difference"
)

---

## 👥 Phase 4: Multi-Agent System (MAS)

From the lecture: *"A multi-agent system consists of multiple AI agents working collectively."*

Let's build a simple MAS where:
- **Researcher Agent** → searches for information
- **Analyst Agent** → processes and summarizes findings

They collaborate by passing messages!

In [ ]:
# ============================================
# MULTI-AGENT SYSTEM — Two agents collaborating
# ============================================

class SpecializedAgent:
    def __init__(self, name, role, instructions):
        self.name = name
        self.role = role
        self.instructions = instructions
        self.inbox = []  # Messages from other agents

    def receive_message(self, from_agent, message):
        """Receive a message from another agent"""
        self.inbox.append({"from": from_agent, "message": message})
        print(f"   📨 {self.name} received message from {from_agent}")

    def process(self, task):
        """Process a task using the agent's role and any messages received"""
        context = ""
        if self.inbox:
            context = "\nMessages from other agents:\n"
            for msg in self.inbox:
                context += f"- From {msg['from']}: {msg['message'][:500]}\n"

        prompt = f"""You are {self.name}, a {self.role}.
{self.instructions}

Task: {task}
{context}
Respond concisely and professionally."""

        response = model.generate_content(prompt)
        return response.text

# Create two specialized agents
researcher = SpecializedAgent(
    name="Researcher",
    role="Research Specialist",
    instructions="Your job is to gather information and facts about a topic. Be thorough but concise."
)

analyst = SpecializedAgent(
    name="Analyst",
    role="Data Analyst",
    instructions="Your job is to take research findings and produce a clear, actionable summary with insights."
)

print("✅ Multi-Agent System created!")
print(f"   Agent 1: {researcher.name} ({researcher.role})")
print(f"   Agent 2: {analyst.name} ({analyst.role})")

In [ ]:
# ============================================
# RUN THE MULTI-AGENT WORKFLOW
# ============================================

topic = "The current state of Agentic AI frameworks and their key differences"

print(f"\n{'='*60}")
print(f"🎯 MULTI-AGENT TASK: {topic}")
print(f"{'='*60}")

# Step 1: Researcher gathers information
print("\n📚 Step 1: Researcher is working...")
research_output = researcher.process(f"Research this topic: {topic}")
print(f"\n{researcher.name}'s findings:")
print(f"{'─'*40}")
print(research_output[:800])

# Step 2: Pass findings to the Analyst
print(f"\n{'─'*40}")
print("\n📨 Step 2: Passing findings to Analyst...")
analyst.receive_message("Researcher", research_output)

time.sleep(2)  # Rate limit

# Step 3: Analyst produces summary
print("\n📊 Step 3: Analyst is processing...")
analysis_output = analyst.process(
    f"Analyze the research findings about '{topic}' and produce a summary with key insights and recommendations."
)
print(f"\n{analyst.name}'s Analysis:")
print(f"{'─'*40}")
print(analysis_output)

print(f"\n{'='*60}")
print("✅ Multi-agent collaboration complete!")

---

## 🏆 Phase 5: Mini Challenge!

Now it's YOUR turn! Choose one of these challenges:

### Challenge A: Add a New Tool
Add a `translator` tool to the agent that translates text between languages.

### Challenge B: Add Self-Correction
Modify the agent to detect when a tool fails and retry with a different approach.

### Challenge C: Three-Agent Team
Create a 3-agent system: Researcher → Writer → Reviewer

---

**Pick one and implement it below!**